# CachePilot 当前进度 Colab 验收

这个 notebook 验证当前仓库已经实现的 CPU 控制面和 FastAPI Gateway：

- 安装固定的 FastAPI、测试和 lint 依赖；
- 检查 Python、Git、模型配置和可见 GPU；
- 运行全量 pytest 与 Ruff；
- 使用标准 TestClient 验证普通 JSON、SSE、tenant 隔离、取消、配额和非法请求 reservation 不增长；
- 在 GPU runtime 中可选加载固定 revision 的 Qwen，并验收 TorchExecutor 单请求生成；
- 验证 VllmExecutor 适配边界，并在锁定版本可安装时执行真实 vLLM stream/abort/usage 验收；
- 启动真正的 uvicorn HTTP 进程，再验证健康检查和普通请求。

当前默认后端是确定性 CPU Echo 后端。它用于 API/SSE/生命周期验收，**不等于真实 Qwen GPU 推理**；TorchExecutor 已实现单请求路径，VllmExecutor 已实现公开异步引擎适配，但两者都需要单独安装 GPU 依赖并显式注入。

## 1. 获取仓库

如果你在自己的 fork 中运行，只需要修改 REPO_URL。重复执行本单元不会删除已有目录。

In [ ]:
import os
import pathlib
import shutil
import subprocess

REPO_URL = "https://github.com/honey-floria/CachePilot.git"
REPO_REF = ""  # 可填 branch、tag 或 commit；留空使用默认分支
WORKDIR = pathlib.Path("/content/CachePilot")

if not (WORKDIR / ".git").exists():
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)

if REPO_REF:
    subprocess.run(["git", "fetch", "--all", "--tags"], cwd=WORKDIR, check=True)
    subprocess.run(["git", "checkout", REPO_REF], cwd=WORKDIR, check=True)

os.chdir(WORKDIR)
print("工作目录:", pathlib.Path.cwd())
print(subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True).stdout.strip())

In [ ]:
import json
import platform
import sys

print("Python:", sys.version.replace("\n", " "))
print("Platform:", platform.platform())
expected_python = (3, 13, 15)
if sys.version_info[:3] != expected_python:
    print("WARNING: 项目基线是 Python 3.13.15；本次 Colab 结果不作为该基线的正式证据。")
else:
    print("Python baseline: PASS")

model_config = json.loads(pathlib.Path("config/model.json").read_text())
print("Configured model:", model_config["model_id"])
print("Model revision:", model_config["model_revision"])
print("Service context limit:", model_config["initial_service_context_limit"])

gpu = subprocess.run(["bash", "-lc", "command -v nvidia-smi >/dev/null && nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader || true"], capture_output=True, text=True)
print("Visible GPU:")
print(gpu.stdout.strip() or "none (current Gateway smoke test does not require GPU)")

## 2. 安装 CPU/API 验收依赖

Colab 的 Python 小版本可能暂时不是项目锁定的 3.13.15。为避免把 pip 的 Python 版本门禁误当成 Gateway 失败，这里安装同样固定版本的运行时和测试工具，并通过 PYTHONPATH 直接运行仓库代码。

In [ ]:
%pip install -q --disable-pip-version-check \
  fastapi==0.115.12 \
  httpx==0.28.1 \
  pydantic==2.11.4 \
  pytest==8.3.5 \
  ruff==0.9.10 \
  starlette==0.46.2 \
  uvicorn==0.34.2

In [ ]:
import os
import sys

os.environ["PYTHONPATH"] = str(WORKDIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
print("PYTHONPATH:", os.environ["PYTHONPATH"])
import fastapi
import httpx
print("FastAPI:", fastapi.__version__)
print("httpx:", httpx.__version__)


## 3. 全量静态检查和测试

In [ ]:
subprocess.run([sys.executable, "-m", "ruff", "check", "."], cwd=WORKDIR, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=WORKDIR, check=True)
print("STATIC_AND_TESTS=PASS")

## 4. 空服务骨架探活

这一步保留原有 colab_acceptance 的标准库空服务验收，确认基础 HTTP 进程仍可启动。

In [ ]:
import time
import urllib.request

EMPTY_PORT = 8765
empty_server = subprocess.Popen(
    [sys.executable, "-m", "cachepilot.runtime.empty_service", "--host", "127.0.0.1", "--port", str(EMPTY_PORT)],
    cwd=WORKDIR,
    env={**os.environ, "PYTHONPATH": str(WORKDIR)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def empty_probe(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{EMPTY_PORT}{path}", timeout=5) as response:
        return response.status, response.read().decode()

try:
    for _ in range(50):
        if empty_server.poll() is not None:
            raise RuntimeError(empty_server.stdout.read())
        try:
            health_status, health_body = empty_probe("/healthz")
            if health_status == 200:
                break
        except OSError:
            time.sleep(0.1)
    else:
        raise TimeoutError("empty service did not become healthy")
    ready_status, ready_body = empty_probe("/readyz")
    metrics_status, metrics_body = empty_probe("/metrics")
    assert json.loads(health_body)["status"] == "ok"
    assert ready_status == 200 and json.loads(ready_body)["status"] == "ready"
    assert metrics_status == 200 and "cachepilot_empty_service_up 1" in metrics_body
    print("EMPTY_SERVICE_HEALTH=PASS")
    print("EMPTY_SERVICE_READY=PASS")
    print("EMPTY_SERVICE_METRICS=PASS")
finally:
    empty_server.terminate()
    try:
        empty_server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        empty_server.kill()
        empty_server.wait()


## 5. Gateway API / SSE 验收

这里不依赖公网模型下载，使用项目默认的确定性 CPU 后端。SSE 检查按标准 data: 行解析，并确认最后是 [DONE]。

In [ ]:
import json
from fastapi.testclient import TestClient

from cachepilot.gateway.api import create_app

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
app = create_app()
client = TestClient(app)

def chat_body(stream=True):
    return {
        "model": MODEL,
        "messages": [{"role": "user", "content": "Hello from Colab"}],
        "stream": stream,
        "max_tokens": 16,
    }

assert client.get("/healthz").status_code == 200
assert client.get("/readyz").json()["status"] == "ready"

normal = client.post(
    "/v1/chat/completions",
    headers={"X-Tenant-ID": "team-a", "X-Request-ID": "colab-normal"},
    json=chat_body(stream=False),
)
assert normal.status_code == 200, normal.text
assert normal.json()["object"] == "chat.completion"
assert normal.json()["choices"][0]["message"]["role"] == "assistant"

with client.stream(
    "POST",
    "/v1/chat/completions",
    headers={
        "X-Tenant-ID": "team-a",
        "X-Request-ID": "colab-stream",
        "Accept": "text/event-stream",
    },
    json=chat_body(stream=True),
) as response:
    assert response.status_code == 200, response.text
    assert response.headers["content-type"].startswith("text/event-stream")
    lines = [line for line in response.iter_lines() if line]

data_lines = [line for line in lines if line.startswith("data: ")]
assert data_lines[-1] == "data: [DONE]"
chunks = [json.loads(line.removeprefix("data: ")) for line in data_lines[:-1]]
assert chunks[0]["choices"][0]["delta"]["role"] == "assistant"
assert "usage" in chunks[-1]

print("NORMAL_JSON=PASS")
print("SSE_PARSE=PASS")
print("SSE_CHUNKS:", len(chunks), "DONE: PASS")

In [ ]:
from cachepilot.runtime.registry import RequestNotFoundError

runtime = app.state.gateway_runtime
before = runtime.admission.snapshot()
invalid = chat_body(stream=True)
invalid["tools"] = []
bad = client.post(
    "/v1/chat/completions",
    headers={"X-Tenant-ID": "team-a", "X-Request-ID": "colab-invalid"},
    json=invalid,
)
after = runtime.admission.snapshot()
assert bad.status_code == 400
assert bad.json()["error"]["code"] == "unknown_field"
assert before.reserved_blocks == after.reserved_blocks == 0
assert before.active_sequences == after.active_sequences == 0
try:
    runtime.registry.get("colab-invalid")
except RequestNotFoundError:
    pass
else:
    raise AssertionError("invalid request entered Registry")

hidden = client.get(
    "/v1/requests/colab-normal",
    headers={"X-Tenant-ID": "team-b"},
)
assert hidden.status_code == 404

print("INVALID_NO_RESERVATION=PASS")
print("TENANT_ISOLATION=PASS")
print("METRICS:\n", client.get("/metrics").text)

In [ ]:
cancel_app = create_app()
cancel_client = TestClient(cancel_app)
cancel_runtime = cancel_app.state.gateway_runtime
cancel_runtime.prepare(
    chat_body(stream=True),
    {"X-Tenant-ID": "team-a", "X-Request-ID": "colab-cancel"},
)
first = cancel_client.post(
    "/v1/requests/colab-cancel/cancel",
    headers={"X-Tenant-ID": "team-a"},
)
second = cancel_client.post(
    "/v1/requests/colab-cancel/cancel",
    headers={"X-Tenant-ID": "team-a"},
)
assert first.status_code == 202
assert second.status_code == 200
assert cancel_runtime.admission.snapshot().reserved_blocks == 0
print("CANCEL_FIRST=202")
print("CANCEL_REPEAT=200")
print("CANCEL_RESERVATION_RELEASE=PASS")

## 6. 真实 HTTP 进程探活

上一节使用 TestClient；本节再启动 uvicorn 子进程，通过 TCP HTTP 访问 /healthz、/readyz 和普通聊天接口。

In [ ]:
import time
import urllib.request

PORT = 8766
server = subprocess.Popen(
    [sys.executable, "-m", "cachepilot.gateway.api", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd=WORKDIR,
    env={**os.environ, "PYTHONPATH": str(WORKDIR)},
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

def get_json(path):
    with urllib.request.urlopen(f"http://127.0.0.1:{PORT}{path}", timeout=5) as response:
        return response.status, json.loads(response.read())

try:
    for _ in range(50):
        if server.poll() is not None:
            raise RuntimeError(server.stdout.read())
        try:
            status, health = get_json("/healthz")
            if status == 200:
                break
        except OSError:
            time.sleep(0.1)
    else:
        raise TimeoutError("Gateway did not become healthy")

    ready_status, ready = get_json("/readyz")
    request_data = json.dumps(chat_body(stream=False)).encode()
    request = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/chat/completions",
        data=request_data,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "X-Tenant-ID": "team-a",
            "X-Request-ID": "colab-http",
        },
    )
    with urllib.request.urlopen(request, timeout=10) as response:
        chat_status = response.status
        chat_payload = json.loads(response.read())
    assert health["status"] == "ok"
    assert ready_status == 200 and ready["status"] == "ready"
    assert chat_status == 200 and chat_payload["object"] == "chat.completion"
    print("TCP_HEALTH=PASS")
    print("TCP_READY=PASS")
    print("TCP_CHAT=PASS")
finally:
    server.terminate()
    try:
        server.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()


## 7. TorchExecutor GPU 单请求验收（可选）

Colab 需要在“运行时类型”中选择 GPU。该单元加载 `config/model.json` 固定的 Qwen revision，验证 TorchExecutor 的单请求路径；它不验证 batching 或性能。


In [ ]:
if not shutil.which("nvidia-smi"):
    print("TORCH_GPU_ACCEPTANCE=SKIP (请在 Colab 选择 GPU runtime)")
else:
    # Colab 当前公开软件源未必提供项目 GPU 基线中的未来版本。
    # 4.48.3 已支持 Qwen2.5；让 Transformers 自行选择兼容的 tokenizers。
    install = subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--upgrade",
            "transformers==4.48.3", "safetensors>=0.4.3",
        ],
        text=True,
        capture_output=True,
    )
    if install.returncode != 0:
        print(install.stdout)
        print(install.stderr)
        raise RuntimeError("TorchExecutor dependencies could not be installed")
    print(install.stdout[-2000:])
    import torch
    from cachepilot.config.baseline import load_model_baseline
    from cachepilot.executors import TorchExecutor, TorchExecutorConfig
    from cachepilot.gateway.contracts import validate_chat_completion_request

    baseline = load_model_baseline(WORKDIR / "config" / "model.json")
    assert torch.cuda.is_available(), "CUDA is not available"
    print("Torch:", torch.__version__)
    print("CUDA:", torch.version.cuda, torch.cuda.get_device_name(0))
    torch_executor = TorchExecutor(
        TorchExecutorConfig(
            model_id=baseline.model_id,
            tokenizer_id=baseline.tokenizer_id,
            model_revision=baseline.model_revision,
            tokenizer_revision=baseline.tokenizer_revision,
            device="cuda",
            dtype=("bfloat16" if torch.cuda.is_bf16_supported() else "float16"),
            context_limit=baseline.service_context_limit,
            trust_remote_code=baseline.trust_remote_code,
        )
    )
    torch_request = validate_chat_completion_request(
        {
            "model": baseline.model_id,
            "messages": [{"role": "user", "content": "Reply with exactly: torch-ok"}],
            "stream": False,
            "max_tokens": 8,
        },
        {"X-Tenant-ID": "team-a", "X-Request-ID": "colab-torch-1"},
        configured_model=baseline.model_id,
    )

    async def collect_torch_output():
        return [item.text async for item in torch_executor.generate(torch_request)]

    pieces = await collect_torch_output()
    output = "".join(pieces)
    assert output.strip(), "TorchExecutor returned empty output"
    assert torch_executor.supports_batching is False
    print("TORCH_OUTPUT:", output)
    print("TORCH_SINGLE_REQUEST=PASS")
    print("TORCH_BATCHING=NOT_IMPLEMENTED_BY_DESIGN")

## 8. VllmExecutor 适配边界验收

这一步不安装 vLLM，通过 fake async engine 验证 request ID、delta stream、abort、usage 和 Gateway SSE 对接。


In [ ]:
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/unit/test_vllm_executor.py",
        "tests/integration/test_vllm_gateway.py",
    ],
    cwd=WORKDIR,
    check=True,
)
print("VLLM_ADAPTER_ACCEPTANCE=PASS")

## 9. 真实 vLLM GPU 验收（可选）

该单元尝试安装 `config/dependencies.json` 锁定的 vLLM 版本，然后在**全新的 Python 子进程**中验证真实 delta stream、request ID、usage 和 abort。子进程会把 pip 安装的 NVIDIA 动态库目录加入 `LD_LIBRARY_PATH`，避免当前 notebook 内核看不到 `libcudart.so.13`。安装会占用较长时间；若锁定 wheel、CUDA runtime 或 Colab 驱动不兼容，单元会明确打印 `SKIP` 原因；其他未知错误会打印 `FAIL` 并抛出异常。建议在全新的 GPU runtime 中运行。


In [ ]:
import site
import textwrap

if not shutil.which("nvidia-smi"):
    print("VLLM_REAL_GPU=SKIP (请在 Colab 选择 GPU runtime)")
else:
    dependency_config = json.loads((WORKDIR / "config" / "dependencies.json").read_text())
    locked_vllm = dependency_config["profiles"]["vllm_executor"]["vllm"]
    print("Locked vLLM version:", locked_vllm)
    install_vllm = subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "--upgrade",
            f"vllm=={locked_vllm}",
        ],
        text=True,
        capture_output=True,
    )
    if install_vllm.returncode != 0:
        print(install_vllm.stdout[-4000:])
        print(install_vllm.stderr[-4000:])
        print("VLLM_REAL_GPU=SKIP (锁定版本在当前 Colab 软件源不可安装)")
    else:
        print(install_vllm.stdout[-2000:])
        site_roots = {pathlib.Path(path) for path in site.getsitepackages()}
        site_roots.add(pathlib.Path(site.getusersitepackages()))
        library_dirs = set()
        for site_root in site_roots:
            library_dirs.update(
                path for path in site_root.glob("nvidia/**/lib") if path.is_dir()
            )
            torch_lib = site_root / "torch" / "lib"
            if torch_lib.is_dir():
                library_dirs.add(torch_lib)
        library_dirs = sorted(str(path) for path in library_dirs)
        libcudart = [
            str(pathlib.Path(path) / "libcudart.so.13")
            for path in library_dirs
            if (pathlib.Path(path) / "libcudart.so.13").exists()
        ]
        print("CUDA library directories:", len(library_dirs))
        print("libcudart.so.13:", libcudart or "not found")

        smoke_env = os.environ.copy()
        smoke_env["PYTHONPATH"] = str(WORKDIR) + os.pathsep + smoke_env.get("PYTHONPATH", "")
        smoke_env["LD_LIBRARY_PATH"] = os.pathsep.join(
            library_dirs + ([smoke_env["LD_LIBRARY_PATH"]] if smoke_env.get("LD_LIBRARY_PATH") else [])
        )
        smoke_env["PYTHONUNBUFFERED"] = "1"
        smoke_env["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

        vllm_smoke_script = textwrap.dedent(
            f"""
            import asyncio
            import inspect
            import pathlib
            from importlib import metadata

            from cachepilot.config.baseline import load_model_baseline
            from cachepilot.executors import VllmExecutor, VllmExecutorConfig
            from cachepilot.gateway.contracts import validate_chat_completion_request

            LOCKED_VLLM = {locked_vllm!r}
            WORKDIR = pathlib.Path({str(WORKDIR)!r})

            def vllm_request(baseline, request_id, content, max_tokens):
                return validate_chat_completion_request(
                    {{
                        "model": baseline.model_id,
                        "messages": [{{"role": "user", "content": content}}],
                        "stream": True,
                        "max_tokens": max_tokens,
                    }},
                    {{"X-Tenant-ID": "team-a", "X-Request-ID": request_id}},
                    configured_model=baseline.model_id,
                )

            async def main():
                assert metadata.version("vllm") == LOCKED_VLLM
                baseline = load_model_baseline(WORKDIR / "config" / "model.json")
                executor = VllmExecutor(
                    VllmExecutorConfig(
                        model_id=baseline.model_id,
                        tokenizer_id=baseline.tokenizer_id,
                        model_revision=baseline.model_revision,
                        tokenizer_revision=baseline.tokenizer_revision,
                        dtype="float16",
                        context_limit=baseline.service_context_limit,
                        gpu_memory_utilization=0.75,
                        trust_remote_code=baseline.trust_remote_code,
                        expected_vllm_version=LOCKED_VLLM,
                    )
                )
                try:
                    normal = vllm_request(
                        baseline, "colab-vllm-1", "Reply with exactly: vllm-ok", 8
                    )
                    pieces = [item async for item in executor.generate(normal)]
                    output = "".join(item.text for item in pieces)
                    usage = executor.usage("colab-vllm-1")
                    assert output.strip(), "VllmExecutor returned empty output"
                    assert usage is not None
                    assert usage.request_id == "colab-vllm-1"
                    assert usage.completion_tokens == sum(item.token_count for item in pieces)
                    assert usage.total_tokens == usage.prompt_tokens + usage.completion_tokens

                    cancel = vllm_request(
                        baseline, "colab-vllm-cancel", "Write a long numbered list.", 64
                    )
                    stream = executor.generate(cancel)
                    first_delta = await anext(stream)
                    await executor.cancel("colab-vllm-cancel", reason="explicit")
                    remaining = [item async for item in stream]
                    assert executor.cancel_reasons["colab-vllm-cancel"] == "explicit"
                    assert not remaining, "tokens were delivered after abort"

                    print("VLLM_OUTPUT:", output)
                    print("VLLM_USAGE:", usage)
                    print("VLLM_FIRST_DELTA_BEFORE_ABORT:", first_delta.text)
                    print("VLLM_REAL_GPU=PASS")
                    print("VLLM_BATCHING_OWNER=vllm")
                finally:
                    shutdown = getattr(executor.engine, "shutdown_background_loop", None)
                    if callable(shutdown):
                        result = shutdown()
                        if inspect.isawaitable(result):
                            await result

            asyncio.run(main())
            """
        )

        try:
            smoke = subprocess.run(
                [sys.executable, "-c", vllm_smoke_script],
                cwd=WORKDIR,
                env=smoke_env,
                text=True,
                capture_output=True,
                timeout=1200,
            )
        except subprocess.TimeoutExpired as exc:
            print((exc.stdout or "")[-4000:])
            print((exc.stderr or "")[-4000:])
            print("VLLM_REAL_GPU=SKIP (真实引擎启动或推理超过 20 分钟)")
        else:
            print(smoke.stdout[-8000:])
            if smoke.returncode == 0:
                assert "VLLM_REAL_GPU=PASS" in smoke.stdout
            else:
                print(smoke.stderr[-8000:])
                diagnostics = smoke.stdout + "\n" + smoke.stderr
                environment_markers = (
                    "libcudart.so.13",
                    "CUDA driver version is insufficient",
                    "forward compatibility was attempted",
                    "no kernel image is available",
                )
                if any(marker in diagnostics for marker in environment_markers):
                    print("VLLM_REAL_GPU=SKIP (锁定 vLLM 的 CUDA 13 runtime 与当前 Colab GPU/驱动不兼容)")
                else:
                    print("VLLM_REAL_GPU=FAIL")
                    raise RuntimeError(
                        "真实 vLLM 子进程失败；请查看上方 stderr 末尾"
                    )


## 10. 当前结论

如果前面的单元全部成功，本次结果代表：

- 当前 Gateway 的普通 JSON 和 SSE 协议可用；
- 非法请求不会进入 Registry 或占用 reservation；
- tenant 查询隔离、显式取消和重复取消语义可用；
- uvicorn 真实 HTTP 进程可启动并响应。

如果 GPU 单元显示 `TORCH_SINGLE_REQUEST=PASS`，则证明固定 Qwen revision 已经通过 TorchExecutor 单请求生成。`VLLM_ADAPTER_ACCEPTANCE=PASS` 证明适配边界正确；只有 `VLLM_REAL_GPU=PASS` 才证明锁定版本的真实 vLLM stream、usage 和 abort 已在当前 GPU 环境运行。以上结果仍不代表 Torch batching、物理 KV 可观测或性能基准已经完成。

In [ ]:
print("COLAB_GATEWAY_ACCEPTANCE=PASS")
print("Scope: CPU control plane + FastAPI Gateway + deterministic backend")
print("Real GPU model inference: see TORCH_SINGLE_REQUEST / VLLM_REAL_GPU results above")